# NHCS Scenario 6 — Two doors of the same hospitals: the ED and the ward side by side

_The federation model of the MIP, with two datasets under one Data Model. Both datasets. Level: after one ED and one inpatient scenario. About 60 minutes._

**Before you start.** The NHCS pathology holds two datasets from about 200 US hospitals, 2020 to
2022: 1,368,648 emergency department visit records and 488,636 inpatient discharge records, each a
5% sample of all encounters of the participating hospitals. **A row is an encounter record, not a
patient**, and the two datasets cannot be linked: each is an independent sample, the identifiers
were renumbered, and NCHS merged the ED outcome "admitted" into "other". The MIP does not use the
survey weights: every figure describes *the sampled encounters*, not US hospital care. Codes are
NCHS codes: Sex 1 = Male, 2 = Female; NCHS flags 1 = Yes, 2 = No; curated flags 0 = No, 1 = Yes;
age is top-coded at 86 and the length of stay at 15 days.

**How this notebook works.** Every analysis is a *federated* call: the code selects a dataset and
variables, defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a
model). No record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect. These are the largest datasets of this MIP: the notebook prints how
long the regressions take.

**Learning goals.** Select two datasets at once and read a result with one row per dataset and one
pooled row; explain what the pooled row means in a federation and when it is meaningless; compare
the case mix of two settings; run a test across datasets; and watch the same months in two kinds of
records.

**Background.** A MIP pathology can hold several datasets that share a Data Model: in a hospital
federation, one dataset per hospital or registry, each staying on its own node. Every analysis then
returns one result per dataset and one pooled result computed across them without moving any
record. The NHCS pathology uses this device for two datasets that come from the same 200 hospitals
but hold different records: emergency department visits and inpatient stays. Selecting both shows
the federation at work on 1,857,284 records, the largest analysis of this material, and asks a
clinical question on the way: how does the case mix change between the ED door and the ward? A
pooled figure across the two mixes visits and stays, two different kinds of record, and is shown
here to be understood, not used.

## 1. Connect to the MIP and open the NHCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHCS")        # add version="ED_Inpatient_v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the platform labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

ed = dataset("nhcs_ed", "NHCS emergency department visits, USA, 2020-2022")
ip = dataset("nhcs_ip", "NHCS inpatient discharges, USA, 2020-2022")
display(dm)
display(dm.datasets.to_frame())

In [ ]:
# The variable tree is the anatomy of one encounter record: Encounter, Patient, Discharge,
# Diagnoses (with the curated condition flags and the NCHS CCSR flags), and the branch
# Inpatient stay (empty for ED visits): length of stay and procedures.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one Dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_summary(result, variables, dataset=POOLED):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v, dataset):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable, dataset=POOLED):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable, dataset):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan, "missing": (r.get("data") or {}).get("num_na")})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",), dataset=POOLED):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable, dataset)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None, datasets=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def band_means(numeric, bands, base_filter=None, datasets=None):
    """n, mean and median of a numerical variable in bands given as {label: filter}."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({"band": label, "n": d.get("num_dtps"), "mean": d.get("mean"), "median": d.get("q2")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None, datasets=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f, datasets=datasets).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True, datasets=datasets).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1", datasets=None):
    """% of records with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f, datasets=datasets).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True, datasets=datasets).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def month_label(i):
    """Month index 1..36 -> '2020-01' .. '2022-12'."""
    return f"{2020 + (i - 1) // 12}-{(i - 1) % 12 + 1:02d}"


def by_month(outcomes, variable, months=range(1, 37), datasets=None):
    """% of records with each outcome (code 1) per month of the time axis: one describe per month."""
    rows = []
    for m in months:
        res = analysis(outcomes, F(variable) == m, datasets=datasets).describe()
        row = {"month": m, "label": month_label(m)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get("1", 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None, rotate=0):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:,.0f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(7, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def n_fmt(value):
    # a count with thousands separators, or n/a when the platform did not return it
    return f"{value:,}" if isinstance(value, (int, float)) else "n/a"


def ttest_from_describe(result_a, result_b, variable):
    """The pooled-variance t statistic of two groups from their describe results (n, mean, SD):
    all a t-test needs, for groups the platform's t-test cannot form (several codes pooled)."""
    a = numeric_summary(result_a, [variable]).iloc[0]
    b = numeric_summary(result_b, [variable]).iloc[0]
    n1, n2 = a["n"], b["n"]
    sp2 = ((n1 - 1) * a["sd"] ** 2 + (n2 - 1) * b["sd"] ** 2) / (n1 + n2 - 2)
    t = (a["mean"] - b["mean"]) / math.sqrt(sp2 * (1 / n1 + 1 / n2))
    return {"mean_a": a["mean"], "mean_b": b["mean"], "diff": a["mean"] - b["mean"], "t": t, "n": n1 + n2}


def term_label(term, variables):
    """Model term -> readable label: 'SEX[2]' -> 'Sex = Female', 'AGE' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def linear_table(result, variables):
    """Linear regression result -> one row per term with coefficient (units of the outcome), 95% CI and p."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return pd.DataFrame({"term": [term_label(t, variables) for t in raw.get("indep_vars", [])],
                         "coefficient": raw.get("coefficients"), "ci_low": raw.get("lower_ci"),
                         "ci_high": raw.get("upper_ci"), "p": raw.get("pvalues")})


def show_coefficients(table, title, unit="days"):
    """Table and dot chart of linear-regression coefficients (the intercept left out of the chart)."""
    display(table.style.format({"coefficient": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table[table["term"] != "Intercept"].iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["coefficient"], range(len(t)),
                xerr=[t["coefficient"] - t["ci_low"], t["ci_high"] - t["coefficient"]], fmt="o", capsize=4)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel(f"{unit} added, for the same values of the other predictors")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The dataset and the variables of this scenario

| Code | Label in the MIP | Tree location (under NHCS) | Role | Values |
|---|---|---|---|---|
| `setting` | Care setting | Encounter | which dataset a record belongs to; curation-derived | 1 Inpatient discharge; 2 Emergency department visit |
| `AGE` | Age | Patient | per dataset and pooled; t-test | integer, 0-86 years |
| `SEX` | Sex | Patient | per dataset and pooled | 1 Male; 2 Female |
| `age_group` | Age group | Patient | chi-square across datasets; curation-derived | 1 Under 1 year; 2 1-17 years; 3 18-44 years; 4 45-64 years; 5 65-74 years; 6 75-84 years; 7 85 years and over |
| `n_dx` | Number of diagnoses | Diagnoses | per dataset and pooled; curation-derived | integer, 0-30 |
| `charlson_group` | Charlson index group | Diagnoses / Condition flags (curated) | per dataset and pooled; curation-derived | 0 0 (no Charlson condition); 1 1-2; 2 3-4; 3 5 or more |
| `dx1_chapter` | First-listed diagnosis, ICD-10 chapter | Diagnoses | case mix; curation-derived | 22 codes, listed in the tree |
| `dx1_group` | First-listed diagnosis, group | Diagnoses | case mix; curation-derived | 40 codes, listed in the tree |
| `YEAR` | Survey year | Encounter | COVID-19 by year per setting | 2020 2020; 2021 2021; 2022 2022 |
| `covid_dx` | COVID-19, any position | Diagnoses / Condition flags (curated) | COVID-19 by year per setting; curation-derived | 0 No; 1 Yes |
| `month_index` | Month index 2020-2022 | Encounter | the same months in both settings; curation-derived | integer, 1-36 months |
| `died` | Died during encounter | Discharge | a figure that must not be pooled; curation-derived | 0 No; 1 Yes |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `AGE`) is what the csv
column is called; you will see it in some result payloads. The first line below chooses the
dataset(s) every call of this notebook uses.

In [ ]:
DATASETS = [ed, ip]                      # both datasets: every describe() answers per dataset and pooled

setting    = dm.variables["Care setting"]
age        = dm.variables["Age"]
sex        = dm.variables["Sex"]
age_group  = dm.variables["Age group"]
n_dx       = dm.variables["Number of diagnoses"]
charlson   = dm.variables["Charlson index group"]
chapter    = dm.variables["First-listed diagnosis, ICD-10 chapter"]
group      = dm.variables["First-listed diagnosis, group"]
year       = dm.variables["Survey year"]
covid      = dm.variables["COVID-19, any position"]
month      = dm.variables["Month index 2020-2022"]
died       = dm.variables["Died during encounter"]
display(mip.to_frame([setting, age, sex, age_group, n_dx, charlson, chapter, group, year, covid, month, died]))

## Step 1. One result, three columns

One `describe()` with both datasets selected: the result holds one row per dataset and one pooled
row for every variable.

*Expected:* age, mean (median): ED visits 41.7 (40), inpatient discharges 48.5 (54), all datasets
43.5 (43). Women 53.6%, 54.7%, 54.1%. Number of diagnoses, mean: 4.4, 11.4, 6.2. Charlson index
0: 76.1%, 44.2%, 67.7%. The pooled column is the federation's answer, weighted by the size of each
member; here it averages a visit record and a stay record, which nobody wants.

In [ ]:
t0 = time.perf_counter()
res = analysis([age, sex, n_dx, charlson]).describe()
print(f"describe() on both datasets answered in {time.perf_counter() - t0:.0f} s")
columns = {"ED visits": ed, "Inpatient discharges": ip, "All datasets": POOLED}
rows = {}
for name, ds in columns.items():
    a, d = numeric_summary(res, [age], ds).iloc[0], numeric_summary(res, [n_dx], ds).iloc[0]
    rows[name] = {"records": int(a["n"] + a["missing"]), "age, mean": a["mean"], "age, median": a["median"],
                  "women %": level_percent(res, sex, ("Female",), ds), "diagnoses, mean": d["mean"],
                  "Charlson 0 %": level_percent(res, charlson, ("0 (no Charlson condition)",), ds)}
three = pd.DataFrame(rows).T                       # one row per column of the result, measures as columns
display(three.style.format({"records": "{:,.0f}", "age, mean": "{:.1f}", "age, median": "{:.0f}", "women %": PCT,
                            "diagnoses, mean": "{:.1f}", "Charlson 0 %": PCT}))

## Step 2. The case mix at the door and on the ward

The first-listed diagnosis per dataset, in chapters and in groups.

*Expected, chapters (ED versus inpatient):* symptoms and signs 23.4% versus 2.9%; injury 16.6%
versus 9.3%; respiratory 7.9% versus 5.2%; musculoskeletal 7.4% versus 4.0%; digestive 6.0% versus
8.1%; circulatory 4.7% versus 14.8%; pregnancy and childbirth 2.7% versus 11.0%; factors influencing
health status 3.5% versus 11.9%. *Groups:* other symptoms and signs 14.2% versus 2.1%; other injury
11.6% versus 2.1%; chest pain 4.3% versus 0.3%; sepsis 1.1% versus 5.6%; liveborn infant 0.0%
versus 9.8%; pregnancy and childbirth 2.7% versus 11.0%. The ED is a place of symptoms and
injuries, most of which go home; the ward is a place of established diseases, births and sepsis.

In [ ]:
res = analysis([chapter, group]).describe()
def mix(variable):
    out = pd.DataFrame({name: category_counts(res, variable, ds).set_index("level")["percent"] for name, ds in columns.items()})
    return out.fillna(0).sort_values("ED visits", ascending=False)
chapters, groups = mix(chapter), mix(group)
display(chapters.head(10).style.format(PCT))
display(groups.loc[[l for l in groups.index if any(k in l for k in ("symptoms", "Other injury", "Chest pain", "Sepsis", "Liveborn", "Pregnancy"))]].style.format(PCT))
ax = chapters.head(10)[["ED visits", "Inpatient discharges"]].iloc[::-1].plot(kind="barh", figsize=(8, 4.5), color=["#4C72B0", "#C44E52"])
ax.set_xlabel("% of records"); ax.set_ylabel(""); ax.set_title("First-listed diagnosis chapter: the ED door and the ward")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Step 3. A test across datasets

The variable Care setting says which dataset a record belongs to. With both datasets selected,
the *Chi-squared Test* of the age group by setting and the *Independent t-test* of age by setting
(groups 1 = inpatient, 2 = ED) compare the two members in one call each.

*Expected:* chi-square 147,332, p < 0.001: infants are 11.2% of stays (newborns included)
against 2.2% of visits; 65 and over 35.5% of stays against 20.5% of visits. Mean age 48.5 against
41.7 years, a difference of 6.8 years, t about 165.

In [ ]:
chi = analysis([setting, age_group], drop_missing=True).chi_square_test(x=setting, y=age_group)
print(f"Age group by setting: chi2 = {chi.raw['chi2']:,.0f}, p = {fmt_p(chi.raw['p_value'])}")
t = analysis([age, setting], drop_missing=True).t_test(variable=age, group_by=setting, group_a="1", group_b="2")
s = ttest_summary(t)
print(f"Age, inpatient minus ED: {s['mean_diff']:.1f} years, t = {s['t_stat']:.0f}, p = {fmt_p(s['p'])}, n = {n_fmt(s['n_obs'])}")
shares = pd.DataFrame({name: category_counts(analysis([age_group]).describe(), age_group, ds).set_index("level")["percent"] for name, ds in columns.items()})
display(shares.style.format(PCT))

## Step 4. COVID-19 by year in each setting

With both datasets selected, a filter on the setting selects one member; selecting that dataset
alone gives the same numbers.

*Expected:* ED visits 2.6%, 4.6%, 4.7% in 2020, 2021 and 2022 (chi-square 2,981); inpatient stays
4.9%, 5.4%, 5.7% (chi-square 103). Both p < 0.001. COVID-19 is more frequent on the ward than at
the door, and its share grew in both settings while the acute waves subsided: in 2022 the code
often marks an incidental infection in a patient admitted for something else.

In [ ]:
by_setting = {}
for name, c in (("ED visits", "2"), ("Inpatient discharges", "1")):
    t = crosstab(year, covid, base_filter=F(setting) == c, event_label="with COVID-19")
    by_setting[name] = t.set_index("Survey year")["% with COVID-19"]
    print(f"{name}: chi2 = {t.attrs['chi2']:,.0f}, p = {fmt_p(t.attrs['p'])}")
check = crosstab(year, covid, event_label="with COVID-19", datasets=[ed]).set_index("Survey year")["% with COVID-19"]
by_setting["ED visits, dataset selected alone"] = check
display(pd.DataFrame(by_setting).style.format(PCT))

## Step 5. The same months in two kinds of records

A histogram of the month index per setting, each curve set to 100 in January 2020.

*Expected:* in April 2020 ED visits fell to 47% of their January 2020 level, inpatient discharges
to 70%. By mid-2021 visits were back to about 90 to 100% and discharges to 115 to 125%; both curves
hold those levels in 2022. The pandemic emptied the emergency department far more than the wards,
which filled with COVID-19 patients; the changing hospital sample explains part of the levels after
2020.

In [ ]:
curves = {}
for name, c in (("ED visits", "2"), ("Inpatient discharges", "1")):
    h = analysis([month], F(setting) == c, drop_missing=True).histogram(variable=month, bins=36).raw["histogram"][0]
    counts = pd.Series([v or 0 for v in h["counts"]], index=[month_label(i) for i in range(1, 37)])
    curves[name] = 100 * counts / counts.iloc[0]
curves = pd.DataFrame(curves)
display(curves.iloc[[0, 3, 6, 12, 18, 24, 30, 35]].style.format("{:.0f}"))
ax = curves.plot(marker="o", figsize=(10, 3.4), color=["#4C72B0", "#C44E52"])
ax.axhline(100, color="k", linewidth=0.8, linestyle="--")
ax.set_ylabel("January 2020 = 100"); ax.set_title("Monthly volume of the two kinds of records")
ax.set_xticks(range(0, 36, 3)); ax.set_xticklabels(curves.index[::3], rotation=45, fontsize=8)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Step 6. A figure that must not be pooled

*Expected:* 0.5% of ED visits died, 2.6% of inpatient discharges, 1.1% of all records. "1.1%
died" is a number without a meaning: a death in the ED and a death on the ward are counted on
different denominators. A pooled result is only as sensible as the question asked of the members;
the MIP computes it, the analyst decides whether it exists.

In [ ]:
res = analysis([died]).describe()
print(", ".join(f"{name}: {level_percent(res, died, ('Yes',), ds):.2f}% died" for name, ds in columns.items()))

## Step 7. State the result

A model answer:

> With both NHCS datasets selected, the MIP returns every result per dataset and pooled across
> 1.86 million records. The two settings describe the same hospitals from two doors: emergency
> visits are younger (42 against 49 years), carry 3 diagnoses against 10, are three quarters free
> of Charlson conditions against under half, and are coded with symptoms and injuries, where the
> ward is coded with circulatory disease, childbirth and sepsis. COVID-19 was more frequent among
> stays than among visits and grew in both from 2020 to 2022; in April 2020 visits fell to half of
> their January level and discharges to 70%. Pooled figures across the two datasets, such as a 1.1%
> death rate, have no meaning because the records are not of one kind; in a federation of like
> datasets they are the point.

**Questions for discussion.** In a federation of five hospitals' stroke registries, which pooled
figures would you want, and which per-hospital figures would you refuse to pool? Which questions
does the missing link between the two datasets forbid, and which analyses remain honest? Chest pain
is 4.3% of visits and 0.3% of stays: what does that ratio say about the ED's work? Why is the
pooled mean age closer to the ED's than to the ward's?